# 01 — eFeature Extraction

Build an `EModelEFeatureExtractionScanConfig`, expand it via `GridScanGenerationTask`,
and run the `EModelEFeatureExtractionTask` for each coordinate. The task downloads
each `ElectricalCellRecording`'s NWB asset from entitycore and extracts e-features
through [BluePyEModel](https://github.com/openbraininstitute/BluePyEModel)'s
`extract_save_features_protocols` (which wraps `bluepyefe` + eFEL), run through a
local access point. A minimal `config/recipes.json` is written, but no
`EModel_pipeline`, morphology or mechanisms are needed at this stage.

**Reads from:** the entitycore staging project (`ElectricalCellRecording` entities).

**Writes to:** `obi-output/01_efeature_extraction/grid_scan/0/` — a working
directory containing the downloaded NWB files under `ephys_data/<entity-id>/`,
the minimal `config/recipes.json` and `config/extract_config/targets.json`, the
extraction `figures/`, and `extracted_features.json` (the serialised
`FitnessCalculatorConfiguration`). The optimisation stage in notebook 02 picks
the features file up and slots it into `config/features/<emodel>.json`.

## Imports

In [ ]:
from pathlib import Path

import obi_one as obi
from entitysdk import Client, ProjectContext
from obi_auth import get_token
from obi_one.core.info import Info
from obi_one.scientific.tasks.emodel_optimization.task1_efeature_extraction.blocks import (
    ExtractionInitialize,
    Settings,
)


## Connect to entitycore staging

The extraction task downloads the recording NWB assets from entitycore. We use the
staging environment + test project bundled with `obi_one`. The first call to
`get_token(environment="staging")` opens a browser tab for authentication.


In [ ]:
token = get_token(environment="staging")
project_context = ProjectContext(
    virtual_lab_id=obi.LAB_ID_STAGING_TEST,
    project_id=obi.PROJECT_ID_STAGING_TEST,
)
db_client = Client(
    api_url="https://staging.cell-a.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)
print("Connected to entitycore staging.")

## Build the scan config

Every `bluepyefe` parameter relevant to extraction is exposed via blocks.
The default `efeatures_by_protocol.protocols` is the L5PC mirror
(IDthresh/IDrest/IV/APWaveform/sAHP with a curated `extract=True` selection).
Step amplitudes and stimulus timing live in the NWB and are read by the task
at execution time, so neither needs to be set here.

Key settings:
- `compute_rheobase=True` — estimates each cell's rheobase from the protocol
  specified in `rheobase_protocol_name` (here `IDThreshold`).
- `validation_protocols` — protocols held out from optimisation (used only for validation).

In [ ]:
# A few arbitrary ElectricalCellRecording entities from the staging test project.
RECORDING_IDS = (
    # "492bdec5-2dce-4ae0-8b85-f020a1ad1d92",
    # "812a8721-1681-49a2-a155-59ab30981079",
    "00854004-4390-4a42-bf9d-5e672e8c8484",
    "66fd126c-f478-47e9-88ad-d12f7f27c84f"
)

scan_config = obi.EModelEFeatureExtractionScanConfig(
    info=Info(
        campaign_name="L5PC eFeature Extraction",
        campaign_description="Extract e-features from staging test recordings.",
    ),
    initialize=ExtractionInitialize(
        electrical_cell_recording=tuple(
            obi.ElectricalCellRecordingFromID(id_str=rid) for rid in RECORDING_IDS
        ),
    ),
    settings=Settings(
        compute_rheobase=True,
        rheobase_protocol_name="IDThreshold",
        validation_protocols="sAHP",  # hold out sAHP for validation
    ),
)
print("Settings:")
print(f"  compute_rheobase: {scan_config.settings.compute_rheobase}")
print(f"  rheobase_protocol_name: {scan_config.settings.rheobase_protocol_name}")
print(f"  validation_protocols: {scan_config.settings.validation_protocols}")
print(f"  default_std_value: {scan_config.settings.default_std_value}")
print()
print("efeatures_by_protocol:")
print(f"  threshold_based: {scan_config.efeatures_by_protocol.threshold_based}")
print(f"  autoselect: {scan_config.efeatures_by_protocol.autoselect}")

## Inspect the recordings' protocols

Use the `/declared/electrical-cell-recording-protocols` endpoint to discover
the protocol (ecode) names stored in each recording's `stimuli` metadata.
The endpoint returns both per-recording and the union across all requested
recordings. We then use this union to populate
``scan_config.efeatures_by_protocol.protocols`` instead of the L5PC defaults.

In [ ]:
import httpx

OBI_ONE_URL = "http://127.0.0.1:8100"

response = httpx.get(
    f"{OBI_ONE_URL}/declared/electrical-cell-recording-protocols",
    params={"recording_ids": list(RECORDING_IDS)},
    headers={"Authorization": f"Bearer {token}"},
    timeout=30,
)
response.raise_for_status()
data = response.json()

by_recording = data["by_recording"]
protocol_union = data["union"]

print("Per-recording protocols:")
for rid, prots in by_recording.items():
    print(f"  {rid}: {len(prots)} protocols")
print(f"\nUnion ({len(protocol_union)}): {protocol_union}")

## Drive `efeatures_by_protocol.protocols` from the entity metadata

For every protocol name in the union, look it up in `PROTOCOL_CATALOGUE`
and instantiate the matching :class:`Protocol` subclass. Flip `extract=True`
on every feature each matched protocol can extract so bluepyefe has targets
to chew on. Protocols not in the catalogue are skipped (they have no
matching bluepyefe eCode).

Note: `compute_rheobase=True` in settings will use the protocol specified in
`rheobase_protocol_name` automatically — no separate rheobase block is needed.

In [ ]:
from obi_one.scientific.tasks.emodel_optimization.task1_efeature_extraction.protocols_and_features import (
    EFeature,
    PROTOCOL_CATALOGUE,
)

catalogue_by_name = {p_cls.name: p_cls for p_cls in PROTOCOL_CATALOGUE}
matched_names = [n for n in protocol_union if n in catalogue_by_name]
skipped_names = [n for n in protocol_union if n not in catalogue_by_name]

configured_protocols = []
for name in matched_names:
    instance = catalogue_by_name[name]()
    for fname in type(instance).model_fields:
        attr = getattr(instance, fname)
        if isinstance(attr, EFeature):
            attr.extract = True
    configured_protocols.append(instance)

scan_config.efeatures_by_protocol.protocols = tuple(configured_protocols)

print(f"Matched {len(matched_names)} protocols from the union: {matched_names}")
print(f"Skipped {len(skipped_names)} (no matching Protocol class): {skipped_names}")
for p in scan_config.efeatures_by_protocol.protocols:
    print(f"  {p.name}: {len(p.selected_efeatures())} features extracting")

## Sanity-check the amplitude reader

The task auto-discovers per-protocol step amplitudes from each NWB so the
user never has to type them. Download one recording's NWB, run the reader
against the matched protocols, and confirm the values look reasonable
(BBP NWBs typically expose currents in nA, so step amplitudes for IV are
around -0.1–0 nA, APWaveform around 0.2–0.3 nA, etc.).


In [ ]:
import tempfile
from pathlib import Path

from obi_one.scientific.library.electrical_cell_recording_properties import (
    _read_amplitudes_from_nwb,
)

sample_recording = obi.ElectricalCellRecordingFromID(id_str=RECORDING_IDS[0])
with tempfile.TemporaryDirectory() as tmp:
    nwb_path = sample_recording.download_asset(dest_dir=Path(tmp), db_client=db_client)
    amps = _read_amplitudes_from_nwb(nwb_path, matched_names)

print(f"Discovered amplitudes (nA) in {sample_recording.id_str}:")
for proto, amp_list in amps.items():
    print(f"  {proto}: {amp_list}")


## Run the grid scan

Per-coordinate output goes to `obi-output/01_efeature_extraction/grid_scan/<idx>/`
(the `ZERO_INDEX` directory option keeps the path stable for downstream notebooks).


In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/01_efeature_extraction/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute(db_client=db_client)

obi.run_tasks_for_generated_scan(grid_scan, db_client=db_client)


## Inspect the extracted features

In [ ]:
import json
from pathlib import Path

coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)
print()

features_path = coord_root / "extracted_features.json"
print("Features:", features_path)
features = json.loads(features_path.read_text())
print("Top-level keys:", list(features))
print(
    f"Extracted {len(features.get('efeatures', []))} efeatures across"
    f" {len(features.get('protocols', []))} protocols."
)

## Inspect the recipe (validation_protocols)

The extraction task writes a `recipes.json` that carries `validation_protocols`
forward to Workflow A. Confirm they're present.

In [ ]:
recipes_path = coord_root / "config" / "recipes.json"
recipes = json.loads(recipes_path.read_text())
ps = recipes["emodel"]["pipeline_settings"]
print("Pipeline settings in recipe:")
print(f"  validation_protocols: {ps.get('validation_protocols')}")
print(f"  extract_absolute_amplitudes: {ps.get('extract_absolute_amplitudes')}")
print(f"  efel_settings: {ps.get('efel_settings')}")
print(f"  name_Rin_protocol: {ps.get('name_Rin_protocol')}")
print(f"  name_rmp_protocol: {ps.get('name_rmp_protocol')}")
